# 04 — End to end: generate → evaluate → export

The generation and export steps run fully offline. The evaluate step needs a live
Azure model plus the model dependencies, so it is skipped with a message when no
credentials are configured.

## 1. Generate — an adversarial testset from the curated bank (offline)

In [ ]:
from llminspector.generation import AdversarialGenerator

gen = AdversarialGenerator.from_excel(
    "../tests/test_sample/test_adversarialdata.xlsx",
    capability="all",
    sample_size=5,
)
seed_result = await gen.a_generate()
print(f"Generated {len(seed_result.goldens)} adversarial goldens.")
seed_result.to_pandas().head()

## 2. Your system answers the seeds

Stubbed here so the flow is self-contained; in practice you would call your LLM
application for each seed prompt.

`Golden.to_test_case()` carries the golden's `id` across as `golden_id`, so every
scored row stays traceable to the golden it came from.

In [ ]:
from llminspector.dataset import EvaluationDataset

answered = EvaluationDataset(
    test_cases=[
        g.to_test_case(actual_output="(stubbed answer)") for g in seed_result.goldens
    ]
)
len(answered)

## 3. Evaluate — only when a live model is configured

In [ ]:
import os

from llminspector import a_evaluate, reporting

have_creds = all(
    os.getenv(f"LLMINSPECTOR_{k}")
    for k in ("AZURE_ENDPOINT", "API_VERSION", "API_KEY")
)

if have_creds:
    from llminspector.config import AzureSettings
    from llminspector.metrics import AnswerJailbreakMetric, SentimentMetric
    from llminspector.models import AzureOpenAIModel

    model = AzureOpenAIModel(AzureSettings.from_env())
    metrics = [
        SentimentMetric(model, target="actual_output"),
        AnswerJailbreakMetric(model),
    ]
    result = await a_evaluate(answered, metrics)
    display(result.to_pandas())
    print(reporting.summary(result))

    result.to_excel("/tmp/llminspector_e2e_eval.xlsx")
    print("Wrote /tmp/llminspector_e2e_eval.xlsx")
else:
    print("No Azure credentials found — skipping the evaluate step.")

## 4. Export the generated seed set (always)

In [ ]:
gen.to_excel("/tmp/llminspector_e2e_seeds.xlsx")
print("Wrote /tmp/llminspector_e2e_seeds.xlsx")